Students implement GBFS and A* algorithms following TODO 1 - 2. \
Students can add supporting attributes and methods to the two classes as needed.

# Libraries

In [1]:
import os
import heapq

# Graph class

In [2]:
# Directed, weighted graphs
class Graph:
  def __init__(self):
    self.AL = dict() # adjacency list
    self.V = 0
    self.E = 0
    self.H = dict()

  def __str__(self):
    res = 'V: %d, E: %d\n'%(self.V, self.E)
    for u, neighbors in self.AL.items():
      line = '%d: %s\n'%(u, str(neighbors))
      res += line
    for u, h in self.H.items():
      line = 'h(%d) = %d\n'%(u, h)
      res += line
    return res

  def print(self):
    print(str(self))

  def load_from_file(self, filename):
    '''
        Example input file:
            V E
            u v w
            u v w
            u v w
            ...
            u1 h1
            u2 h2
            u3 h3
            ...

        # input.txt
        7 8
        0 1 5
        0 2 6
        1 3 12
        1 4 9
        2 5 5
        3 5 8
        3 6 7
        4 6 4
        0 14
        1 13
        2 12
        3 11
        4 10
        5 9
        6 0
    '''
    if os.path.exists(filename):
      with open(filename) as g:
        self.V, self.E = [int(it) for it in g.readline().split()]
        for i in range(self.E):
          line = g.readline()
          u, v, w = [int(it) for it in line.strip().split()]
          if u not in self.AL:
            self.AL[u] = []
          self.AL[u].append((v, w))
        for i in range(self.V):
          line = g.readline()
          u, h = [int(it) for it in line.strip().split()]
          self.H[u] = h

In [3]:
g = Graph()
g.load_from_file('input.txt')
g.print()

V: 7, E: 8
0: [(1, 5), (2, 6)]
1: [(3, 12), (4, 9)]
2: [(5, 5)]
3: [(5, 8), (6, 7)]
4: [(6, 4)]
h(0) = 14
h(1) = 13
h(2) = 12
h(3) = 11
h(4) = 10
h(5) = 9
h(6) = 0



# Search Strategies

In [4]:
class BestSearchStrategy:
  def search(self, g: Graph, src: int, dst: int) -> tuple:
    expanded = [] # list of expanded vertices in the traversal order
    path = [] # path from src to dst
    return expanded, path

In [5]:
class GBFS(BestSearchStrategy):
    def search(self, g: Graph, src: int, dst: int) -> tuple:
        expanded = []   # list of expanded vertices in the traversal order
        path = []       # path from src -> dst

        # --- GBFS uses only h(n) as priority (ignores actual path cost) ---

        frontier = [(g.H[src], src)]  # Min-heap priority queue : (h(n), node)
        parent = {src : None}          # Track where each node came from
        visited = set()               # Set of already-expanded nodes

        while frontier :
            _, node = heapq.heappop(frontier)  # Always expand the node with smallest h(n)

            if node in visited :
                continue  # Already expanded → skip (handles duplicate entries in heap)

            expanded.append(node)
            visited.add(node)

            if node == dst :
                break  # Goal reached → stop

            # Expand neighbors
            if node in g.AL :
                for neighbor, _ in g.AL[node] :   # Edge weight is ignored in GBFS
                    if neighbor not in visited :
                        parent[neighbor] = node
                        heapq.heappush(frontier, (g.H[neighbor], neighbor))  # Priority = h(n) only

        # Reconstruct path by tracing back through parent map
        if dst in parent :
            node = dst
            while node is not None :
                path.append(node)
                node = parent[node]
            path.reverse()

        return expanded, path

In [6]:
class AStar(BestSearchStrategy):
    def search(self, g: Graph, src: int, dst: int) -> tuple:
        expanded = []   # list of expanded vertices in the traversal order
        path = []       # path from src -> dst

        # --- A* uses f(n) = g(n) + h(n) as priority ---
        # g(n) = actual cost from src to node n
        # h(n) = heuristic estimated cost from n to dst
        # f(n) = total estimated cost of the cheapest path through n

        frontier = [(g.H[src], 0, src)]  # Min-heap : (f(n), g(n), node)
        g_cost = {src : 0}                # Best known g(n) for each node
        parent = {src : None}             # Track where each node came from

        while frontier :
            _, cost, node = heapq.heappop(frontier)  # Always expand node with smallest f(n)

            if node in expanded :
                continue  # Already expanded → skip (stale heap entry)

            expanded.append(node)

            if node == dst :
                break  # Goal reached → stop

            # Expand neighbors
            if node in g.AL :
                for neighbor, weight in g.AL[node] :
                    new_g = cost + weight                    # Tentative g(neighbor)
                    f_value = new_g + g.H[neighbor]         # f(neighbor) = g + h

                    # Only push if this path to neighbor is cheaper than any known path
                    if neighbor not in g_cost or new_g < g_cost[neighbor] :
                        g_cost[neighbor] = new_g
                        parent[neighbor] = node
                        heapq.heappush(frontier, (f_value, new_g, neighbor))

        # Reconstruct path by tracing back through parent map
        if dst in parent :
            node = dst
            while node is not None :
                path.append(node)
                node = parent[node]
            path.reverse()

        return expanded, path

# Evaluation

In [7]:
gbfs = GBFS()
astar = AStar()

for stg in [gbfs, astar]:
  print(stg)
  expanded, path = stg.search(g, 0, g.V-1)
  print(expanded)
  print(path)

[0, 2, 5, 1, 4, 6]
[0, 1, 4, 6]
[0, 1, 2, 5, 4, 6]
[0, 1, 4, 6]


# Submission

*   Students download the notebook after completion
*   Rename the notebook in which inserting your student ID at the beginning. \
For example, **123456-lec05-BestFirstSearch-HW.ipynb**
*   Finally, submit the file